# Gold model

Esta notebook construye el modelo dimensional solicitado en la capa Gold.

Tablas objetivo:

- `dim_fecha`
- `dim_canal`
- `dim_instrumento`
- `dim_cliente`
- `fact_operaciones`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
SOURCE_TABLE = "byma.silver.operaciones_clean"

DIM_FECHA_TABLE = "byma.gold.dim_fecha"
DIM_CANAL_TABLE = "byma.gold.dim_canal"
DIM_INSTRUMENTO_TABLE = "byma.gold.dim_instrumento"
DIM_CLIENTE_TABLE = "byma.gold.dim_cliente"
FACT_OPERACIONES_TABLE = "byma.gold.fact_operaciones"

In [0]:
df = spark.table(SOURCE_TABLE)

display(df.limit(10))
df.printSchema()

fecha,tipoTran,id_cliente,descripcion_titulo,moneda,simbolo_titulo,cantidad,precio,id_transaccion,origen,_ingested_at,_source_file,_pipeline_version,fecha_partition,fecha_operacion,monto_total,es_fin_de_semana,flag_outlier_monto
2026-01-02T21:55:45.000Z,Compra,CLI9C5D59E9,YPF,ARS,YPFD,3.000000,55137.628400,TXNid25owf75935a,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-02,2026-01-02,165412.885200000,false,false
2026-01-02T13:32:05.000Z,Compra,CLIADDE511A,Bono Rep. Argentina Usd Step Up 2030,USD,AL30D,14.000000,0.668900,TXNr6k2vmwwyzx4w,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-02,2026-01-02,9.364600000,false,false
2026-01-02T16:02:19.000Z,Compra,CLIAAF9DF77,Ecogas Inversiones S.A.,ARS,ECOG,24.000000,3225.509100,TXNa78rapkianouo,Sitio Web Responsive,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-02,2026-01-02,77412.218400000,false,false
2026-01-02T20:25:56.000Z,Compra,CLIDAB0BA0E,Sociedad Comercial del Plata,ARS,COME,15.000000,59.841800,TXN2gqu7dknncptk,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-02,2026-01-02,897.627000000,false,false
2026-01-02T23:35:44.000Z,Compra,CLI76C377AB,Cedear Spdr S&P 500,USD,SPYD,24.000000,35.157900,TXNfqla09lzrk79u,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-02,2026-01-02,843.789600000,false,false
2026-01-02T17:37:15.000Z,Venta,CLI54E2C248,Bono Rep. Argentina Usd Step Up 2030,ARS,AL30,20.000000,1008.444300,TXN5d0p92k38p6fv,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-02,2026-01-02,20168.886000000,false,false
2026-01-02T21:01:21.000Z,Venta,CLICF867CAB,Cedear Microsoft Corp.,USD,MSFTD,2.000000,16.217400,TXNh3wqa1vrgss8k,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-02,2026-01-02,32.434800000,false,false
2026-01-02T17:25:36.000Z,Compra,CLI02CD48BF,Cedear Berkshire Hathaway Inc.,ARS,BRKB,1.000000,35162.293300,TXN4ns7nvv28ue1m,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-02,2026-01-02,35162.293300000,false,false
2026-01-02T21:37:13.000Z,Compra,CLIA315D3EC,Bbva,ARS,BBAR,2.000000,9282.977900,TXNw4vnzvzn2hloi,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-02,2026-01-02,18565.955800000,false,false
2026-01-02T21:37:08.000Z,Compra,CLI3792F458,Autopistas del Sol,ARS,AUSO,6.000000,4072.465700,TXN5aiekx6mj53yx,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-02,2026-01-02,24434.794200000,false,false


root
 |-- fecha: timestamp (nullable = true)
 |-- tipoTran: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- descripcion_titulo: string (nullable = true)
 |-- moneda: string (nullable = true)
 |-- simbolo_titulo: string (nullable = true)
 |-- cantidad: decimal(20,6) (nullable = true)
 |-- precio: decimal(20,6) (nullable = true)
 |-- id_transaccion: string (nullable = true)
 |-- origen: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _pipeline_version: string (nullable = true)
 |-- fecha_partition: date (nullable = true)
 |-- fecha_operacion: date (nullable = true)
 |-- monto_total: decimal(38,9) (nullable = true)
 |-- es_fin_de_semana: boolean (nullable = true)
 |-- flag_outlier_monto: boolean (nullable = true)



In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS byma.gold")

DataFrame[]

## dim_fecha

In [0]:
dim_fecha = (
    df
    .select("fecha_operacion")
    .distinct()
    .withColumn("fecha_sk", F.date_format("fecha_operacion", "yyyyMMdd").cast("int"))
    .withColumn("anio", F.year("fecha_operacion"))
    .withColumn("mes", F.month("fecha_operacion"))
    .withColumn("dia", F.dayofmonth("fecha_operacion"))
    .withColumn("dia_semana", F.dayofweek("fecha_operacion"))
)

(
    dim_fecha
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_FECHA_TABLE)
)

display(dim_fecha.limit(10))

fecha_operacion,fecha_sk,anio,mes,dia,dia_semana
2026-01-02,20260102,2026,1,2,6
2026-01-03,20260103,2026,1,3,7
2026-01-05,20260105,2026,1,5,2
2026-01-06,20260106,2026,1,6,3
2026-01-07,20260107,2026,1,7,4
2026-01-08,20260108,2026,1,8,5
2026-01-09,20260109,2026,1,9,6
2026-01-10,20260110,2026,1,10,7
2026-01-12,20260112,2026,1,12,2
2026-01-13,20260113,2026,1,13,3


## dim_canal

In [0]:
window_canal = Window.orderBy("origen")

dim_canal = (
    df
    .select("origen")
    .distinct()
    .withColumn("canal_sk", F.row_number().over(window_canal))
)

(
    dim_canal
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_CANAL_TABLE)
)

display(dim_canal)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


origen,canal_sk
API,1
App Mobile,2
IOLnet,3
Sitio Web Desktop,4
Sitio Web Responsive,5


## dim_instrumento

In [0]:
window_instrumento = Window.orderBy("simbolo_titulo", "descripcion_titulo", "moneda")

dim_instrumento = (
    df
    .select("simbolo_titulo", "descripcion_titulo", "moneda")
    .distinct()
    .withColumn("instrumento_sk", F.row_number().over(window_instrumento))
    .select(
        "instrumento_sk",
        "simbolo_titulo",
        "descripcion_titulo",
        "moneda"
    )
)

(
    dim_instrumento
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_INSTRUMENTO_TABLE)
)

display(dim_instrumento.limit(20))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


instrumento_sk,simbolo_titulo,descripcion_titulo,moneda
1,#MAV150560106,pagare vto. 18/05/2026,ARS
2,A3,Matba Rofex S.A.,ARS
3,AAL,Cedear American Airlines Group,ARS
4,AALC,Cedear American Airlines Group,USD
5,AALD,Cedear American Airlines Group,USD
6,AAP,Cedear Advance Auto Parts Inc,ARS
7,AAPL,Cedear Apple Inc.,ARS
8,AAPLC,Cedear Apple Inc.,USD
9,AAPLD,Cedear Apple Inc.,USD
10,ABBV,Cedear Abbvie Inc.,ARS


## dim_cliente

In [0]:
window_cliente = Window.orderBy("id_cliente")

dim_cliente = (
    df
    .select("id_cliente", "origen")
    .groupBy("id_cliente")
    .agg(
        F.count("*").alias("cantidad_operaciones"),
        F.first("origen").alias("canal_preferido")
    )
    .withColumn(
        "segmento_actividad",
        F.when(F.col("cantidad_operaciones") >= 10, F.lit("alto"))
         .when(F.col("cantidad_operaciones") >= 3, F.lit("medio"))
         .otherwise(F.lit("bajo"))
    )
    .withColumn("cliente_sk", F.row_number().over(window_cliente))
    .withColumn("valid_from", F.lit("2026-01-01").cast("date"))
    .withColumn("valid_to", F.lit(None).cast("date"))
    .withColumn("is_current", F.lit(True))
    .select(
        "cliente_sk",
        "id_cliente",
        "segmento_actividad",
        "canal_preferido",
        "valid_from",
        "valid_to",
        "is_current"
    )
)

(
    dim_cliente
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_CLIENTE_TABLE)
)

display(dim_cliente.limit(20))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


cliente_sk,id_cliente,segmento_actividad,canal_preferido,valid_from,valid_to,is_current
1,CLI0001BB38,bajo,Sitio Web Desktop,2026-01-01,null,true
2,CLI0003E67E,bajo,App Mobile,2026-01-01,null,true
3,CLI00040D07,bajo,App Mobile,2026-01-01,null,true
4,CLI000439E2,medio,App Mobile,2026-01-01,null,true
5,CLI00070505,bajo,App Mobile,2026-01-01,null,true
6,CLI00083DE8,bajo,Sitio Web Desktop,2026-01-01,null,true
7,CLI0008F7D4,bajo,Sitio Web Desktop,2026-01-01,null,true
8,CLI000B995A,bajo,App Mobile,2026-01-01,null,true
9,CLI000E1CEB,bajo,App Mobile,2026-01-01,null,true
10,CLI000E989E,bajo,App Mobile,2026-01-01,null,true


## fact_operaciones

In [0]:
dim_fecha_df = spark.table(DIM_FECHA_TABLE)
dim_canal_df = spark.table(DIM_CANAL_TABLE)
dim_instrumento_df = spark.table(DIM_INSTRUMENTO_TABLE)
dim_cliente_df = spark.table(DIM_CLIENTE_TABLE).filter(F.col("is_current") == True)

fact_operaciones = (
    df
    .join(dim_fecha_df, df.fecha_operacion == dim_fecha_df.fecha_operacion, "left")
    .join(dim_canal_df, df.origen == dim_canal_df.origen, "left")
    .join(
        dim_instrumento_df,
        (df.simbolo_titulo == dim_instrumento_df.simbolo_titulo) &
        (df.descripcion_titulo == dim_instrumento_df.descripcion_titulo) &
        (df.moneda == dim_instrumento_df.moneda),
        "left"
    )
    .join(dim_cliente_df, df.id_cliente == dim_cliente_df.id_cliente, "left")
    .select(
        df.id_transaccion,
        "fecha_sk",
        "canal_sk",
        "instrumento_sk",
        "cliente_sk",
        df.fecha,
        df.tipoTran,
        df.cantidad,
        df.precio,
        df.monto_total,
        df.es_fin_de_semana,
        df.flag_outlier_monto
    )
)

(
    fact_operaciones
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FACT_OPERACIONES_TABLE)
)

fact_count = spark.table(FACT_OPERACIONES_TABLE).count()

print(f"Registros persistidos en fact_operaciones: {fact_count}")

display(fact_operaciones.limit(20))

Registros persistidos en fact_operaciones: 100000


id_transaccion,fecha_sk,canal_sk,instrumento_sk,cliente_sk,fecha,tipoTran,cantidad,precio,monto_total,es_fin_de_semana,flag_outlier_monto
TXNid25owf75935a,20260102,2,1440,35146,2026-01-02T21:55:45.000Z,Compra,3.000000,55137.628400,165412.885200000,false,false
TXNr6k2vmwwyzx4w,20260102,2,49,39048,2026-01-02T13:32:05.000Z,Compra,14.000000,0.668900,9.364600000,false,false
TXNa78rapkianouo,20260102,5,402,38386,2026-01-02T16:02:19.000Z,Compra,24.000000,3225.509100,77412.218400000,false,false
TXN2gqu7dknncptk,20260102,2,304,49204,2026-01-02T20:25:56.000Z,Compra,15.000000,59.841800,897.627000000,false,false
TXNfqla09lzrk79u,20260102,2,1146,26744,2026-01-02T23:35:44.000Z,Compra,24.000000,35.157900,843.789600000,false,false
TXN5d0p92k38p6fv,20260102,2,47,19131,2026-01-02T17:37:15.000Z,Venta,20.000000,1008.444300,20168.886000000,false,false
TXNh3wqa1vrgss8k,20260102,2,868,46750,2026-01-02T21:01:21.000Z,Venta,2.000000,16.217400,32.434800000,false,false
TXN4ns7nvv28ue1m,20260102,2,202,689,2026-01-02T17:25:36.000Z,Compra,1.000000,35162.293300,35162.293300000,false,false
TXNw4vnzvzn2hloi,20260102,2,131,36593,2026-01-02T21:37:13.000Z,Compra,2.000000,9282.977900,18565.955800000,false,false
TXN5aiekx6mj53yx,20260102,2,103,12508,2026-01-02T21:37:08.000Z,Compra,6.000000,4072.465700,24434.794200000,false,false


## Dimensión Cliente (SCD Tipo 2)

# SCD Type 2 - dim_cliente

Aca simulamos cambios históricos de segmento para clientes, manteniendo versiones con vigencia temporal.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
DIM_CLIENTE_TABLE = "byma.gold.dim_cliente"

In [0]:
dim_cliente = spark.table(DIM_CLIENTE_TABLE)

display(dim_cliente.limit(10))

cliente_sk,id_cliente,segmento_actividad,canal_preferido,valid_from,valid_to,is_current
1,CLI0001BB38,bajo,Sitio Web Desktop,2026-01-01,null,true
2,CLI0003E67E,bajo,App Mobile,2026-01-01,null,true
3,CLI00040D07,bajo,App Mobile,2026-01-01,null,true
4,CLI000439E2,medio,App Mobile,2026-01-01,null,true
5,CLI00070505,bajo,App Mobile,2026-01-01,null,true
6,CLI00083DE8,bajo,Sitio Web Desktop,2026-01-01,null,true
7,CLI0008F7D4,bajo,Sitio Web Desktop,2026-01-01,null,true
8,CLI000B995A,bajo,App Mobile,2026-01-01,null,true
9,CLI000E1CEB,bajo,App Mobile,2026-01-01,null,true
10,CLI000E989E,bajo,App Mobile,2026-01-01,null,true


In [0]:
clientes_scd = (
    dim_cliente
    .filter(F.col("is_current") == True)
    .limit(3)
)

display(clientes_scd)

cliente_sk,id_cliente,segmento_actividad,canal_preferido,valid_from,valid_to,is_current
1,CLI0001BB38,bajo,Sitio Web Desktop,2026-01-01,null,true
2,CLI0003E67E,bajo,App Mobile,2026-01-01,null,true
3,CLI00040D07,bajo,App Mobile,2026-01-01,null,true


In [0]:
clientes_historicos = (
    clientes_scd
    .withColumn("valid_to", F.lit("2026-02-15").cast("date"))
    .withColumn("is_current", F.lit(False))
)

clientes_nuevos = (
    clientes_scd
    .withColumn(
        "segmento_actividad",
        F.when(F.col("segmento_actividad") == "bajo", F.lit("medio"))
         .when(F.col("segmento_actividad") == "medio", F.lit("alto"))
         .otherwise(F.lit("alto"))
    )
    .withColumn("valid_from", F.lit("2026-02-16").cast("date"))
    .withColumn("valid_to", F.lit(None).cast("date"))
    .withColumn("is_current", F.lit(True))
)

dim_cliente_sin_3 = dim_cliente.join(
    clientes_scd.select("id_cliente"),
    on="id_cliente",
    how="left_anti"
)

dim_cliente_scd2 = dim_cliente_sin_3.unionByName(clientes_historicos).unionByName(clientes_nuevos)

display(
    dim_cliente_scd2
    .filter(F.col("id_cliente").isin([row["id_cliente"] for row in clientes_scd.collect()]))
    .orderBy("id_cliente", "valid_from")
)

id_cliente,cliente_sk,segmento_actividad,canal_preferido,valid_from,valid_to,is_current
CLI0001BB38,1,bajo,Sitio Web Desktop,2026-01-01,2026-02-15,false
CLI0001BB38,1,medio,Sitio Web Desktop,2026-02-16,null,true
CLI0003E67E,2,bajo,App Mobile,2026-01-01,2026-02-15,false
CLI0003E67E,2,medio,App Mobile,2026-02-16,null,true
CLI00040D07,3,bajo,App Mobile,2026-01-01,2026-02-15,false
CLI00040D07,3,medio,App Mobile,2026-02-16,null,true


In [0]:
window_cliente_scd = Window.orderBy("id_cliente", "valid_from")

dim_cliente_scd2 = (
    dim_cliente_scd2
    .drop("cliente_sk")
    .withColumn("cliente_sk", F.row_number().over(window_cliente_scd))
    .select(
        "cliente_sk",
        "id_cliente",
        "segmento_actividad",
        "canal_preferido",
        "valid_from",
        "valid_to",
        "is_current"
    )
)

display(
    dim_cliente_scd2
    .filter(F.col("id_cliente").isin([row["id_cliente"] for row in clientes_scd.collect()]))
    .orderBy("id_cliente", "valid_from")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


cliente_sk,id_cliente,segmento_actividad,canal_preferido,valid_from,valid_to,is_current
1,CLI0001BB38,bajo,Sitio Web Desktop,2026-01-01,2026-02-15,false
2,CLI0001BB38,medio,Sitio Web Desktop,2026-02-16,null,true
3,CLI0003E67E,bajo,App Mobile,2026-01-01,2026-02-15,false
4,CLI0003E67E,medio,App Mobile,2026-02-16,null,true
5,CLI00040D07,bajo,App Mobile,2026-01-01,2026-02-15,false
6,CLI00040D07,medio,App Mobile,2026-02-16,null,true


In [0]:
(
    dim_cliente_scd2
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_CLIENTE_TABLE)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
